In [2]:
import numpy as np
import pandas as pd
import torch
import math
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
import csv
import datetime
import matplotlib.pyplot as plt

df = pd.read_csv('merged.csv')
df = df.dropna()
features = ['A','I','c','L','fy','delta_o']
target = 'Pn'
X = df[features].to_numpy()
y = df[target].to_numpy()
noise = np.random.normal(0,1,y.shape)
y = y + noise
SHIFT = 0.01
y_log = np.log(y + SHIFT)
scaler_x = StandardScaler()
X_scaled = scaler_x.fit_transform(X)
scaler_y = StandardScaler()
y_log_scaled = scaler_y.fit_transform(y_log.reshape(-1,1)).ravel()
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_log_scaled, test_size=0.15, random_state=42
)

In [3]:
def perry_robertson(a,i,c,l,fy,d):
    e = 29000
    py = fy*a
    pcre = (math.pi**2)*e*i/(l**2)
    se = pcre/a
    sy = fy
    eta = (d*c)/(i/a)
    val = 0.5*(1+(se/sy)*(eta+1)-math.sqrt((1+(se/sy)*(eta+1))**2-4*(se/sy)))*py
    return val

def physics_informed_loss(y_true,y_pred,x):
    x_real = scaler_x.inverse_transform(x.detach().cpu().numpy())
    pn_real = []
    for row in x_real:
        pn_real.append(perry_robertson(row[0],row[1],row[2],row[3],row[4],row[5]))
    pn_real = np.array(pn_real).reshape(-1,1)
    pn_log = np.log(pn_real + SHIFT)
    pn_scaled = scaler_y.transform(pn_log)
    pn_scaled_t = torch.tensor(pn_scaled,dtype=torch.float32,device=y_pred.device)
    mse_data = nn.MSELoss()(y_pred,y_true)
    mse_phys = nn.MSELoss()(y_pred,pn_scaled_t)
    return mse_data + 0.1*mse_phys

In [4]:
class PINN(nn.Module):
    def __init__(self,in_dim,h_dim,out_dim):
        super(PINN,self).__init__()
        self.fc1 = nn.Linear(in_dim,h_dim)
        self.act1 = nn.LeakyReLU(0.01)
        self.bn1 = nn.BatchNorm1d(h_dim)
        self.dropout1 = nn.Dropout(p=0.2)
        self.fc2 = nn.Linear(h_dim,h_dim)
        self.act2 = nn.LeakyReLU(0.01)
        self.bn2 = nn.BatchNorm1d(h_dim)
        self.dropout2 = nn.Dropout(p=0.2)
        self.fc3 = nn.Linear(h_dim,h_dim)
        self.act3 = nn.LeakyReLU(0.01)
        self.bn3 = nn.BatchNorm1d(h_dim)
        self.dropout3 = nn.Dropout(p=0.2)
        self.fc4 = nn.Linear(h_dim,out_dim)
    def forward(self,x):
        x = self.fc1(x)
        x = self.act1(x)
        x = self.bn1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.act2(x)
        x = self.bn2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        x = self.act3(x)
        x = self.bn3(x)
        x = self.dropout3(x)
        x = self.fc4(x)
        return x

In [1]:
input_dim = 6
hidden_dim = 64
output_dim = 1
model = PINN(input_dim,hidden_dim,output_dim)
optimizer = optim.Adam(model.parameters(),lr=0.01)
header = ['epoch','train_loss','val_loss']
rows = []
train_losses = []
val_losses = []

for epoch in range(1000):
    model.train()
    X_t = torch.tensor(X_train,dtype=torch.float32)
    y_t = torch.tensor(y_train,dtype=torch.float32).unsqueeze(-1)
    optimizer.zero_grad()
    y_p = model(X_t)
    train_loss = physics_informed_loss(y_t,y_p,X_t)
    train_loss.backward()
    optimizer.step()
    model.eval()
    with torch.no_grad():
        X_v = torch.tensor(X_val,dtype=torch.float32)
        y_vt = torch.tensor(y_val,dtype=torch.float32).unsqueeze(-1)
        y_pv = model(X_v)
        val_loss = physics_informed_loss(y_vt,y_pv,X_v)
    rows.append([epoch, train_loss.item(), val_loss.item()])
    train_losses.append(train_loss.item())
    val_losses.append(val_loss.item())

In [6]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs("training_output/normal_1_new/",exist_ok=True)
model_path = f"training_output/normal_1_new/model_{timestamp}.pth"
torch.save(model.state_dict(), model_path)

with open("training_output/normal_1_new/training_log.csv",mode="a",newline="") as f:
    w = csv.writer(f)
    if f.tell()==0:
        w.writerow(header)
    for r in rows:
        w.writerow(r)

In [7]:
plt.figure(figsize=(6,4))
plt.plot(np.arange(len(train_losses)),train_losses,label='Train Loss')
plt.plot(np.arange(len(val_losses)),val_losses,label='Val Loss')
plt.title('Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.savefig('training_output/normal_1_new/loss_curve.png')
plt.close()

In [8]:
import seaborn as sns
plt.figure(figsize=(8,6))
corr_data = df[features + [target]].corr()
sns.heatmap(corr_data, annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap of Features and Pn")
plt.savefig("training_output/normal_1_new/correlation_heatmap.png")
plt.close()